# Notebook 03 - Computer Vision

Goes with **Lesson 03**. We use **OpenCV** (`cv2`), the most popular computer vision library.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
print("OpenCV version:", cv2.__version__)

In [ ]:
from pathlib import Path
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")

## 1. An image is a grid of numbers

In [ ]:
tiny = np.array([
    [  0,   0, 255,   0,   0],
    [  0, 255, 255, 255,   0],
    [255, 255, 255, 255, 255],
    [  0, 255, 255, 255,   0],
    [  0,   0, 255,   0,   0],
], dtype=np.uint8)

print(tiny)
plt.imshow(tiny, cmap="gray")
plt.title("A 5 x 5 image made from numbers")
plt.show()

## 2. A real colour photo

scikit-learn includes a sample photo, so no download is needed.

In [ ]:
from sklearn.datasets import load_sample_image

photo_rgb = load_sample_image("china.jpg")       # RGB order
print("Shape (height, width, channels):", photo_rgb.shape)
print("Total numbers in this photo:", photo_rgb.size)
print("Top-left pixel [R, G, B]:", photo_rgb[0, 0])

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
axes[0].imshow(photo_rgb); axes[0].set_title("colour")
for i, name in enumerate(["Red", "Green", "Blue"]):
    axes[i + 1].imshow(photo_rgb[:, :, i], cmap="gray")
    axes[i + 1].set_title(f"{name} channel")
for ax in axes:
    ax.axis("off")
plt.show()

## 3. Image preprocessing with OpenCV

OpenCV normally uses **BGR** colour order, so we convert carefully.

In [ ]:
photo_bgr = cv2.cvtColor(photo_rgb, cv2.COLOR_RGB2BGR)

gray = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2GRAY)
small = cv2.resize(gray, (160, 107))
blurred = cv2.GaussianBlur(gray, (9, 9), 0)
_, black_white = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
edges = cv2.Canny(gray, 100, 200)

# A CNN-style filter: this 3 x 3 kernel highlights vertical edges
vertical_edge_kernel = np.array([[-1, 0, 1],
                                 [-2, 0, 2],
                                 [-1, 0, 1]], dtype=np.float32)
filtered = cv2.filter2D(gray.astype(np.float32), -1, vertical_edge_kernel)

images = [photo_rgb, gray, blurred, black_white, edges, np.abs(filtered)]
titles = ["original", "grayscale", "blurred", "threshold", "Canny edges", "3x3 vertical-edge filter"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, image, title in zip(axes.ravel(), images, titles):
    ax.imshow(image, cmap=None if image.ndim == 3 else "gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()
print("Resized image shape:", small.shape, " Normalised pixel example:", round(gray[0, 0] / 255, 3))

The last picture is what **one filter** in the first layer of a CNN does. A CNN learns hundreds of such filters by itself.

## 4. Image classification: which digit is it?

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.25, random_state=0)

classifier = SVC(gamma=0.001)
classifier.fit(X_train, y_train)
print(f"Digit classification accuracy: {classifier.score(X_test, y_test):.3f}")

predicted = classifier.predict(X_test[:8])
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, pixels, label in zip(axes, X_test[:8], predicted):
    ax.imshow(pixels.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"says {label}")
    ax.axis("off")
plt.show()

## 5. Object detection: find objects AND where they are

We draw some shapes, then let OpenCV find each one, draw a **bounding box** and name the shape. Real detectors like YOLO do the same for cars, people and animals, using deep learning.

In [ ]:
canvas = np.zeros((300, 500, 3), dtype=np.uint8)
cv2.rectangle(canvas, (30, 40), (150, 140), (255, 255, 255), -1)
cv2.circle(canvas, (260, 90), 55, (255, 255, 255), -1)
triangle = np.array([[400, 30], [470, 150], [330, 150]])
cv2.drawContours(canvas, [triangle], 0, (255, 255, 255), -1)
cv2.rectangle(canvas, (60, 200), (300, 260), (255, 255, 255), -1)

gray_canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
contours, _ = cv2.findContours(gray_canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

result = canvas.copy()
for contour in contours:
    corners = len(cv2.approxPolyDP(contour, 0.03 * cv2.arcLength(contour, True), True))
    name = "triangle" if corners == 3 else "rectangle" if corners == 4 else "circle"
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(result, (x - 5, y - 5), (x + w + 5, y + h + 5), (0, 255, 0), 2)
    cv2.putText(result, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

print("Objects found:", len(contours))
plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title("Detection = label + bounding box")
plt.axis("off")
plt.show()

## 6. Image segmentation: group every pixel

We group the photo's pixels into **K colour groups** with K-means (from Day 1). Every pixel gets a group, like a simple segmentation mask.

In [ ]:
K = 4
pixels = photo_rgb.reshape(-1, 3).astype(np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
_, labels, centres = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

segmented = centres[labels.flatten()].reshape(photo_rgb.shape).astype(np.uint8)
mask = labels.reshape(photo_rgb.shape[:2])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(photo_rgb); axes[0].set_title("original")
axes[1].imshow(segmented); axes[1].set_title(f"{K} colour segments")
axes[2].imshow(mask, cmap="tab10"); axes[2].set_title("segment mask (one colour per group)")
for ax in axes:
    ax.axis("off")
plt.show()

Real segmentation models (U-Net, SAM) group pixels by **object** (road, car, person), not just by colour.

## 7. Face detection (optional - needs your own photo)

Put a clear, front-facing photo at `data/my_photo.jpg` (of yourself, or someone who agrees). OpenCV comes with a ready-made face **detector** (a Haar cascade). It finds **where** faces are. It does **not** know **who** the person is - that would be face **recognition**.

In [ ]:
photo_path = DATA_DIR / "my_photo.jpg"
if not photo_path.exists():
    print("No photo found at", photo_path, "- this step is optional. Add a photo and run again.")
else:
    image = cv2.imread(str(photo_path))
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = detector.detectMultiScale(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), scaleFactor=1.1, minNeighbors=5)
    print("Faces found:", len(faces))
    for (x, y, w, h) in faces:
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 3)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 8. OCR: read text from an image (optional - needs Tesseract)

We create an image with text, then try to read it back. If Tesseract is not installed you will see the install steps (see Lesson 03).

In [ ]:
import os
import shutil

text_image = np.full((120, 600), 255, dtype=np.uint8)
cv2.putText(text_image, "Hello AI Students 2026", (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 1.4, 0, 3)
plt.imshow(text_image, cmap="gray")
plt.axis("off")
plt.show()

try:
    import pytesseract
    windows_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    if not shutil.which("tesseract") and os.path.exists(windows_path):
        pytesseract.pytesseract.tesseract_cmd = windows_path
    print("OCR result:", pytesseract.image_to_string(text_image).strip())
except Exception as error:
    print("OCR is not available:", type(error).__name__)
    print("Install the Tesseract program (Lesson 03, optional section), restart VS Code, and run this cell again.")

## 9. Video analytics (optional - needs a webcam)

A video is many images per second. This cell reads 50 frames from your webcam and counts faces in each one. It is switched off by default. Change `RUN_WEBCAM = False` to `True` to try it. No window opens; it prints how many faces it sees in each frame.

In [ ]:
RUN_WEBCAM = False

if RUN_WEBCAM:
    camera = cv2.VideoCapture(0)
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    for frame_number in range(50):
        ok, frame = camera.read()
        if not ok:
            print("Could not read from the webcam.")
            break
        faces = detector.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.1, 5)
        print(f"frame {frame_number}: {len(faces)} face(s)")
    camera.release()
else:
    print("Webcam demo is switched off. Set RUN_WEBCAM = True to try it.")

**Note:** the installed package `opencv-python` can open windows and webcams. If you only have `opencv-python-headless`, webcam windows do not work.

Go back to **Lesson 03** and do the **Check** section.